# GLM-GAM Model for Hazard Prediction
## スプライン付き一般化加法モデル

このノートブックでは、pyGAMを使用してGLM-GAMモデルを学習し、評価します。

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# プロジェクトのルートディレクトリをパスに追加
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

from data_preprocessing import load_and_preprocess_data
from models.glm_gam import GLMGAMModel
from evaluation import evaluate_regression_model, print_evaluation_results
from utils import save_model, save_figure, save_results

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['MS Gothic', 'Yu Gothic', 'Meiryo']
plt.rcParams['axes.unicode_minus'] = False

## 1. データの読み込みと前処理

In [ ]:
# データ読み込み
print("Loading and preprocessing data...")
data = load_and_preprocess_data(
    target_type='hazard_rate_1to2',
    n_splits=5,
    random_state=42
)

X = data['X']
y = data['y']
groups = data['groups']
fold_splits = data['fold_splits']
feature_info = data['feature_info']

print(f"\nData shape: {X.shape}")
print(f"Target range: [{y.min():.6f}, {y.max():.6f}]")
print(f"Number of unique bridges: {len(np.unique(groups))}")

## 2. 連続特徴量とカテゴリカル特徴量の設定

In [ ]:
# 連続特徴量（スプライン適用）
continuous_features = [
    'Age', '橋長_m', '全幅員_m', '支承数', '伸縮装置数',
    '損傷の進行性_山口市配点', '損傷部位の重要性_山口市配点',
    'LCC_健全度Ⅱ', 'LCC_健全度Ⅲ'
]

# カテゴリカル特徴量（因子項）
categorical_features = [
    '材料区分_encoded', '橋梁形式_encoded', '海岸線区分_encoded',
    '緊急輸送道路_encoded', 'DID地区_encoded', '損傷区分_encoded'
]

# 存在する特徴量のみを使用
continuous_features = [f for f in continuous_features if f in X.columns]
categorical_features = [f for f in categorical_features if f in X.columns]

print(f"Continuous features ({len(continuous_features)}): {continuous_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

## 3. Cross-Validationでモデル学習

In [ ]:
# CV結果を保存
cv_results = []

for fold_idx, (train_idx, test_idx) in enumerate(fold_splits, 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx}/{len(fold_splits)}")
    print(f"{'='*60}")
    
    # データ分割
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # モデル学習
    model = GLMGAMModel(
        continuous_features=continuous_features,
        categorical_features=categorical_features,
        n_splines=10,
        spline_order=3,
        lam=None,  # CV自動選択
        verbose=True
    )
    
    model.fit(X_train, y_train)
    
    # 評価
    results = evaluate_regression_model(
        model, X_train, y_train, X_test, y_test,
        model_name=f'GLM-GAM_fold{fold_idx}'
    )
    results['fold'] = fold_idx
    results['lam'] = model.lam
    cv_results.append(results)
    
    print_evaluation_results(results)

# CV結果のDataFrame化
df_cv_results = pd.DataFrame(cv_results)
print(f"\n{'='*60}")
print("Cross-Validation Summary")
print(f"{'='*60}")
print(df_cv_results[['fold', 'test_rmse', 'test_mae', 'test_r2']].to_string(index=False))
print(f"\nAverage Test RMSE: {df_cv_results['test_rmse'].mean():.6f} ± {df_cv_results['test_rmse'].std():.6f}")
print(f"Average Test MAE:  {df_cv_results['test_mae'].mean():.6f} ± {df_cv_results['test_mae'].std():.6f}")
print(f"Average Test R²:   {df_cv_results['test_r2'].mean():.6f} ± {df_cv_results['test_r2'].std():.6f}")

## 4. 全データでのモデル学習（最終モデル）

In [ ]:
# 全データでモデル学習
print("Training final model on all data...")
final_model = GLMGAMModel(
    continuous_features=continuous_features,
    categorical_features=categorical_features,
    n_splines=10,
    spline_order=3,
    lam=df_cv_results['lam'].median(),  # CVの中央値を使用
    verbose=True
)

final_model.fit(X, y)
final_model.print_summary()

# モデル保存
save_model(final_model, 'glm_gam_final')

## 5. 特徴量の加法的寄与の可視化

In [ ]:
# 連続特徴量の寄与をプロット
fig = final_model.plot_feature_contributions(X, feature_names=continuous_features)
plt.suptitle('GLM-GAM: 連続特徴量の加法的寄与', fontsize=16, fontweight='bold', y=1.02)
save_figure(fig, 'glm_gam_continuous_contributions')
plt.show()

## 6. 結果の保存

In [ ]:
# CV結果を保存
save_results(df_cv_results, 'glm_gam_cv_results')

# 最終モデルのサマリを保存
summary = final_model.get_model_summary()
save_results(summary, 'glm_gam_summary')

print("All results saved successfully!")